Importing necessary functions and libraries for models

In [ ]:
Link to Google Colab file: https://colab.research.google.com/drive/1yuO9WwhVMd2icVRypgdj3vatX3LdG3kq?usp=sharing

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import csv
from sklearn.model_selection import train_test_split, cross_val_score, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

The **load_data** function loads in the .csv file of features and separates feature values from the genre labels in the last column. For the string features, this function separates string / categorical features and encodes them into numerical values using Label Encoder function from SciKit-Learn, then returns all the values in **X_numeric** variable as well as the labels in **y**.

In [ ]:
def load_data(input_file):

  df = pd.read_csv(input_file)

  X = df.iloc[:, :-1].copy()
  y = df.iloc[:, -1].values

  numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
  categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
  label_encoders = {}
  for col in categorical_cols:
      le = LabelEncoder()
      X[col] = le.fit_transform(X[col].astype(str))
      label_encoders[col] = le

  X_numeric = X.values.astype(float)

  return X_numeric, y

This function splits the data into training, validation and testing datasets (80/20), use the function **train_test_split()** from scikit-learn. Then, this function scales features using the **StandardScaler()** function from scikit-learn to make them all within the same scale and not let one over-weight another.



In [ ]:
def split_and_scale_data(X_numeric, y):
  X_train, X_test, y_train, y_test = train_test_split(X_numeric, y, test_size=0.2, random_state=42)
  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)

  return X_train_scaled, X_test_scaled, y_train, y_test

The **train_models()** function trains the inputted ML algorithms (i.e. models) and prints results for each model for accuracy, results of cross validation, and a score for overfitting equivalent to training accuracy - test accuracy, therefore the lower the "overfit" score the better.

In [ ]:
def train_models(models,X_train_scaled, y_train):
  results = {}

  for name, model in models.items():
      print(f"\nTraining {name}...")

      model.fit(X_train_scaled, y_train)

      y_train_pred = model.predict(X_train_scaled)
      y_test_pred = model.predict(X_test_scaled)

      train_acc = accuracy_score(y_train, y_train_pred)
      test_acc = accuracy_score(y_test, y_test_pred)

      cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=10)
      cv_mean = cv_scores.mean()
      cv_std = cv_scores.std()

      results[name] = {
          'train_acc': train_acc,
          'test_acc': test_acc,
          'cv_mean': cv_mean,
          'cv_std': cv_std,
          'overfit': train_acc - test_acc
      }

      print(f" Train Accuracy: {train_acc:.4f}")
      print(f"  Test Accuracy:  {test_acc:.4f}")
      print(f"  CV Accuracy:    {cv_mean:.4f} (+/- {cv_std:.4f})")
      print(f"  Overfitting:    {train_acc - test_acc:.4f}")



Using these functions, I tested the following 5 ML algorithms on the inital "GenreAll.csv" feature and label dataset.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'k-NN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'k-NN (k=10)': KNeighborsClassifier(n_neighbors=10),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

input_file = '/content/drive/MyDrive/GenreAll.csv'
X_numeric, y = load_data(input_file)
X_train_scaled, X_test_scaled, y_train, y_test = split_and_scale_data(X_numeric, y)
train_models(models,X_train_scaled, y_train)




Training Logistic Regression...
 Train Accuracy: 1.0000
  Test Accuracy:  0.8241
  CV Accuracy:    0.8200 (+/- 0.0382)
  Overfitting:    0.1759

Training k-NN (k=5)...
 Train Accuracy: 0.8060
  Test Accuracy:  0.7538
  CV Accuracy:    0.7118 (+/- 0.0364)
  Overfitting:    0.0523

Training k-NN (k=10)...
 Train Accuracy: 0.7620
  Test Accuracy:  0.7387
  CV Accuracy:    0.6852 (+/- 0.0377)
  Overfitting:    0.0233

Training Decision Tree...
 Train Accuracy: 1.0000
  Test Accuracy:  0.6181
  CV Accuracy:    0.6020 (+/- 0.0569)
  Overfitting:    0.3819

Training Random Forest...
 Train Accuracy: 1.0000
  Test Accuracy:  0.8241
  CV Accuracy:    0.8037 (+/- 0.0446)
  Overfitting:    0.1759


**Logistic Regression** and **Random Forest** scored the best in 10-fold cross validation accuracy but scored worse in overfitting, so I proceeded to perform feature selection using the **SelectKBest** method to see if I could improve the overfitting of these 2 models.

In [ ]:
def feature_selection_selectkbest(X_train_scaled, X_test_scaled, y_train, y_test,
                                   models, k_values, c_values=None):

    results = {name: [] for name in models.keys()}

    for k in k_values:
        if k > X_train_scaled.shape[1]:
            continue

        print(f"\nTesting with k={k} features:")

        selector = SelectKBest(f_classif, k=k)
        X_train_selected = selector.fit_transform(X_train_scaled, y_train)
        X_test_selected = selector.transform(X_test_scaled)

        for name, model in models.items():
          if c_values is not None and hasattr(model, 'C'):
            for C in c_values:
              model_clone = model.__class__(**model.get_params())
              model_clone.set_params(C=C)
              model_clone.fit(X_train_selected, y_train)

              train_acc = accuracy_score(y_train, model_clone.predict(X_train_selected))
              test_acc = accuracy_score(y_test, model_clone.predict(X_test_selected))
              cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=10)
              cv_mean = cv_scores.mean()

              results[name].append({
                  'k': k,
                  'C': C,
                  'train': train_acc,
                  'test': test_acc,
                  'cv acc': cv_mean
              })
              print(f"  {name} (C={C}): CV Test={cv_mean:.4f}, Overfit={train_acc-test_acc:.4f}")

          else:

            model_clone = model.__class__(**model.get_params())
            model_clone.fit(X_train_selected, y_train)

            train_acc = accuracy_score(y_train, model_clone.predict(X_train_selected))
            test_acc = accuracy_score(y_test, model_clone.predict(X_test_selected))
            cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=10)
            cv_mean = cv_scores.mean()
            cv_std = cv_scores.std()

            results[name].append({
                'k': k,
                'train': train_acc,
                'test': test_acc,
                'cv acc': cv_mean, '(+/-': cv_std, ')'
                'overfit': train_acc - test_acc,
                'selector': selector  # Save for later use
            })

            print(f"  {name}: CV Test={cv_mean:.4f}, Overfit={train_acc-test_acc:.4f}")

    return results

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'k-NN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'k-NN (k=10)': KNeighborsClassifier(n_neighbors=10),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

input_file = '/content/drive/MyDrive/GenreAll.csv'
k_values = [10, 25, 50]
c_values = None
feature_names= (pd.read_csv(input_file)).columns

X_numeric, y = load_data(input_file)
X_train_scaled, X_test_scaled, y_train, y_test = split_and_scale_data(X_numeric, y)
results = feature_selection_selectkbest(X_train_scaled, X_test_scaled, y_train, y_test, models, k_values, c_values=None)


Testing with k=10 features:
  Logistic Regression: CV Test=0.8200, Overfit=0.0142
  k-NN (k=5): CV Test=0.7118, Overfit=0.0847
  k-NN (k=10): CV Test=0.6852, Overfit=0.0268
  Decision Tree: CV Test=0.6020, Overfit=0.5025
  Random Forest: CV Test=0.8037, Overfit=0.3417

Testing with k=25 features:
  Logistic Regression: CV Test=0.8200, Overfit=0.0183
  k-NN (k=5): CV Test=0.7118, Overfit=0.0334
  k-NN (k=10): CV Test=0.6852, Overfit=0.0107
  Decision Tree: CV Test=0.6020, Overfit=0.4070
  Random Forest: CV Test=0.8037, Overfit=0.2161

Testing with k=50 features:
  Logistic Regression: CV Test=0.8200, Overfit=0.0814
  k-NN (k=5): CV Test=0.7118, Overfit=0.0234
  k-NN (k=10): CV Test=0.6852, Overfit=0.0032
  Decision Tree: CV Test=0.6020, Overfit=0.4121
  Random Forest: CV Test=0.8037, Overfit=0.2010


From these results, I was able to see that using **Logistic Regression** method trained only on the 10 best features (using SelectKBest function from scikit-learn), I was able to reduce the overfitting score down to 1.42%, which were the best results.



In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Logistic Regression (L1 Reg)': LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, random_state=42),
    'Logistic Regression (L2 Reg)': LogisticRegression(penalty='l2', solver='liblinear', max_iter=1000, random_state=42),

}

input_file = '/content/drive/MyDrive/GenreAll.csv'
k_values = [10]
c_values = [0.01, 0.1, 0.5, 1, 10, 100]
feature_names= (pd.read_csv(input_file)).columns

X_numeric, y = load_data(input_file)
X_train_scaled, X_test_scaled, y_train, y_test = split_and_scale_data(X_numeric, y)
results = feature_selection_selectkbest(X_train_scaled, X_test_scaled, y_train, y_test, models, k_values,c_values)


Testing with k=10 features:
  Logistic Regression (C=0.01): CV Test=0.8200, Overfit=-0.0087
  Logistic Regression (C=0.1): CV Test=0.8200, Overfit=-0.0198
  Logistic Regression (C=0.5): CV Test=0.8200, Overfit=0.0092
  Logistic Regression (C=1): CV Test=0.8200, Overfit=0.0142
  Logistic Regression (C=10): CV Test=0.8200, Overfit=0.0230
  Logistic Regression (C=100): CV Test=0.8200, Overfit=0.0230
  Logistic Regression (L1 Reg) (C=0.01): CV Test=0.8087, Overfit=0.0561
  Logistic Regression (L1 Reg) (C=0.1): CV Test=0.8087, Overfit=0.0517
  Logistic Regression (L1 Reg) (C=0.5): CV Test=0.8087, Overfit=0.0103
  Logistic Regression (L1 Reg) (C=1): CV Test=0.8087, Overfit=0.0079
  Logistic Regression (L1 Reg) (C=10): CV Test=0.8087, Overfit=0.0079
  Logistic Regression (L1 Reg) (C=100): CV Test=0.8087, Overfit=0.0091
  Logistic Regression (L2 Reg) (C=0.01): CV Test=0.8035, Overfit=0.0313
  Logistic Regression (L2 Reg) (C=0.1): CV Test=0.8035, Overfit=0.0039
  Logistic Regression (L2 Reg) (

Next I loaded in the **3 second dataset** and the **30 second data set** to see how results changed, but I had to re-write the data load function to ensure no data leakage within the testing set.  

In [ ]:
def load_data_w_song_id(input_file):

  df = pd.read_csv(input_file)

  X = df.iloc[:,1:-1].copy()
  y = df.iloc[:, -1].values
  song_id = df.iloc[:,0].copy()
  song_prefix = song_id.str.rsplit('.', n=1).str[0]

  #convert any of the categorical features into numbers
  numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
  categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
  label_encoders = {}
  for col in categorical_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le
  X = X.values.astype(float)

  feature_names = numeric_cols + categorical_cols

  return X, y, song_prefix.values, feature_names

In [ ]:
def split_and_scale_data_by_group(X, y, song_prefix, test_size=0.2, random_state=42):

  #do group-based splitting
  gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
  train_idx, test_idx = next(gss.split(X, y, groups=song_prefix))

  X_train = X[train_idx]
  X_test = X[test_idx]
  y_train = y[train_idx]
  y_test = y[test_idx]

  train_prefixes = set(song_prefix[train_idx])
  test_prefixes = set(song_prefix[test_idx])
  overlap = train_prefixes.intersection(test_prefixes)

  print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")
  print(f"Train songs: {len(train_prefixes)}, Test songs: {len(test_prefixes)}")

  if overlap:
      print(f"WARNING: {len(overlap)} songs in both sets!")
  else:
      print("✓ No song overlap between sets")

  # Scale
  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)

  return X_train_scaled, X_test_scaled, y_train, y_test


In this iteration, I decided to keep the K best features set at 10 and did not implement a regularizer function as this reduced accuracy.

In [ ]:

k_values = [10]
c_values = None
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'k-NN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'k-NN (k=10)': KNeighborsClassifier(n_neighbors=10),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

input_file = '/content/drive/MyDrive/features_3_sec.csv'

X, y, song_prefixes, feature_names = load_data_w_song_id(input_file)
X_train_scaled, X_test_scaled, y_train, y_test = split_and_scale_data_by_group(X, y, song_prefixes)
print(f"\nResults from {Path(input_file).name}:")
results = feature_selection_selectkbest(X_train_scaled, X_test_scaled, y_train, y_test, models, k_values, c_values)

input_file = '/content/drive/MyDrive/features_30_sec.csv'

X, y, song_prefixes, feature_names = load_data_w_song_id(input_file)
X_train_scaled, X_test_scaled, y_train, y_test = split_and_scale_data_by_group(X, y, song_prefixes)
print(f"\nResults from {Path(input_file).name}:")
results = feature_selection_selectkbest(X_train_scaled, X_test_scaled, y_train, y_test, models, k_values, c_values)


Train samples: 7992, Test samples: 1998
Train songs: 7992, Test songs: 1998
✓ No song overlap between sets

Results from features_3_sec.csv:

Testing with k=10 features:
  Logistic Regression: CV Test=0.6022, Overfit=0.0013
  k-NN (k=5): CV Test=0.5899, Overfit=0.0957
  k-NN (k=10): CV Test=0.5902, Overfit=0.0584
  Decision Tree: CV Test=0.4520, Overfit=0.3846
  Random Forest: CV Test=0.6051, Overfit=0.2419
Train samples: 800, Test samples: 200
Train songs: 800, Test songs: 200
✓ No song overlap between sets

Results from features_30_sec.csv:

Testing with k=10 features:
  Logistic Regression: CV Test=0.6600, Overfit=0.1175
  k-NN (k=5): CV Test=0.5775, Overfit=0.1250
  k-NN (k=10): CV Test=0.5675, Overfit=0.1200
  Decision Tree: CV Test=0.5237, Overfit=0.5637
  Random Forest: CV Test=0.7213, Overfit=0.4587


# Summary of Findings — ML for Music Lab 1

## Tasks and goals

* **Task 1:** Train models on the *GenreAll* (Essentia descriptors) file and maximize classification accuracy. Evaluate a variety of algorithms, check for overfitting, perform feature selection and hyperparameter tuning, and evaluate using 10‑fold cross‑validation.
* **Task 2:** Repeat Task 1 using `features_30_sec.csv` and `features_3_sec.csv`.

All experiments were implemented in the provided Colab notebook. Preprocessing, feature selection, hyperparameter tuning and 10‑fold cross validation were used consistently across experiments.

---

## Methods (pipeline summary)

1. **Data loading & labeling** — datasets loaded with song identifiers; the train/test split was performed *by song* so there is no song overlap between train and test sets.
2. **Preprocessing** — `StandardScaler` was applied to features. Labels were encoded when necessary.
3. **Feature selection** — `SelectKBest` was used to test reduced feature sets (notably `k=10` was evaluated) and reduce overfitting.
4. **Models evaluated**

   * Logistic Regression (with L1/L2 regularization where applicable)
   * k‑Nearest Neighbors (k = 5, 10)
   * Decision Tree
   * Random Forest
   * Tuned model hyperparameters (e.g., `C` for Logistic Regression, `n_neighbors` for k‑NN, and k-values for feature selection).

5. **Evaluation** — primary metric: accuracy. Evaluations reported: train accuracy, test accuracy, 10‑fold CV mean accuracy (with std), and a simple overfitting measure defined as `Train Accuracy − Test Accuracy`.

---

## Key results (compact)

> The numbers below are taken directly from the Colab notebook outputs (10‑fold CV where reported).

### Dataset: `GenreAll` (Essentia descriptors)

* **Logistic Regression**

  * Train accuracy: **1.0000**
  * Test accuracy: **0.8241**
  * 10‑fold CV accuracy: **0.8200** (± 0.0382)
  * Overfitting (Train − Test): **0.1759**

* **k‑NN (k=5)**

  * Train accuracy: **0.8060**
  * Test accuracy: **0.7538**
  * 10‑fold CV accuracy: **0.7118** (± 0.0364)
  * Overfitting (Train - Test): **0.0523**

* **k‑NN (k=10)**

  * Train accuracy: **0.7620**
  * Test accuracy: **0.7387**
  * 10‑fold CV accuracy: **0.6852** (± 0.0377)
  * Overfitting (Train - Test): **0.0233**

* **Decision Tree**

  * Train accuracy: **1.0000**
  * Test accuracy: **0.6181**
  * 10‑fold CV accuracy: **0.6020** (± 0.0569)
  * Overfitting (Train - Test): **0.3819**

* **Random Forest**

  * Train accuracy: **1.0000**
  * Test accuracy: **0.8241**
  * 10‑fold CV accuracy: **0.8037** (± 0.0446)
  * Overfitting: **0.1759**


**Interpretation:** `GenreAll` produced the best results. Logistic Regression and Random Forest reached the highest CV accuracies (≈**82%**), indicating these feature sets contain strong signal for genre classification. The perfect train accuracies suggest the models can fully fit the training samples (potential overfitting risk), but the 10‑fold CV and test accuracies are still reasonably high — suggesting acceptable generalization when using the current data split.

### Dataset: `features_3_sec.csv`

* **Logistic Regression** — 10‑fold CV: **0.6022** (overfit ≈ 0.0013)
* **Random Forest** — 10‑fold CV: **0.6051** (overfit ≈ 0.2419)
* **k‑NN / Decision Tree** — CV roughly in the **0.45–0.59** range; Decision Tree performed worse (CV ≈ **0.4520**).

**Interpretation:** Short 3‑second excerpt features produced lower accuracies (≈60% with best models). This indicates that short excerpts are less informative for the genre labels used (or that the chosen features are noisier at this time scale).

### Dataset: `features_30_sec.csv`

* **Logistic Regression** — 10‑fold CV: **0.6600** (overfit ≈ 0.1175)
* **Random Forest** — 10‑fold CV: **0.6051**
* **k‑NN** — CV ≈ **0.5675–0.5775**
* **Decision Tree** — CV ≈ **0.4520**

**Interpretation:** 30‑second features performed better than 3‑second features but worse than the full Essentia `GenreAll` descriptors. Logistic Regression with selected features reached about **66%** CV accuracy.

---

## Observations about overfitting and model behavior

* Several models (notably Logistic Regression and Random Forest on `GenreAll`) achieved **perfect train accuracy (1.0)**; this is a red flag for overfitting on the training set. However, the 10‑fold CV results (≈0.82) and separate test accuracies (≈0.8241) suggest that while models can memorize training samples, performance on held‑out folds remains strong for the `GenreAll` features.
* Feature selection (`SelectKBest`, e.g., `k=10`) reduced overfitting in several cases. Example: with `k=10`, Logistic Regression's overfitting was reduced to **0.0142** in one experiment.
* Decision trees tended to underperform relative to ensemble methods (Random Forest) and regularized linear models; pure decision trees also showed larger variance and higher overfitting.

---

## What was tried for hyperparameter tuning

* Iterating through different values for parameters such as `C` for Logistic Regression, `n_neighbors` for k‑NN, and `k-values` for SelectKBest features. The notebook reports the CV metrics for the tuned models.

---

## Conclusions and recommendations (next steps)

1. **Best-performing feature set:** `GenreAll` (Essentia descriptors) — best CV ≈ **82%** with Logistic Regression / Random Forest. Use these descriptors as a baseline for future experiments.
2. **Short excerpts (3s) are less informative** than longer segments or the full descriptor set; 30s improves results but does not reach the performance of the full Essentia descriptors.
3. **Address apparent overfitting**:

   * Investigate why training accuracies hit 1.0 (possible data leakage, very expressive models, or label imbalance). Confirm there is truly no leakage (the notebook already splits by song, which is good).
   * Apply stronger regularization, reduce model complexity (or increase `min_samples_leaf` / `max_depth` for trees), or use more aggressive feature selection / PCA.
4. **More experiments** that would likely help:

   * Try `PCA` or other dimensionality reduction (not yet implemented in the notebook) to remove correlated features.
   * Evaluate class balance and consider stratified sampling, class weights, or balanced accuracy if classes are imbalanced.
   * Use more extensive hyperparameter searches (randomized search or larger grid) and nested CV to reduce risk of selection bias.
   * Try ensemble stacking or model blending and more careful calibration for probabilistic outputs.

---

## Quick table (best CV accuracy per dataset)

* `GenreAll` (Essentia descriptors): **~0.82 (82.0%)** — Logistic Regression / Random Forest
* `features_30_sec.csv`: **~0.66 (66.0%)** — Logistic Regression (with feature selection)
* `features_3_sec.csv`: **~0.605 (60.5%)** — Random Forest / Logistic Regression

